# 🌐 GNN: Graph-Aware Neural Network (Complete Pipeline)

**Purpose:** Train a Graph Neural Network that understands road network topology for ambulance ETA prediction

**Data:** Navi Mumbai datasets from Google Drive

**Target:** MAE < 0.12 min (beat old flat GNN)

**Timeline:** Run this in Google Colab (GPU recommended)

## STEP 1: Mount Google Drive & Setup

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
print('✅ Google Drive mounted')

# Set base path to your navi mumbai datasets folder
BASE_PATH = '/content/drive/MyDrive/navi mumbai datasets'

# Check if folder exists
if os.path.exists(BASE_PATH):
    print(f'✅ Found dataset folder: {BASE_PATH}')
    print('\n📁 Contents:')
    for item in os.listdir(BASE_PATH):
        print(f'  - {item}')
else:
    print(f'❌ Folder not found at {BASE_PATH}')
    print('Available paths:')
    for item in os.listdir('/content/drive/MyDrive'):
        print(f'  - {item}')

## STEP 2: Install Required Libraries

In [ ]:
# Install torch-geometric
!pip install torch-geometric -q
!pip install networkx -q

print('✅ All libraries installed')

## STEP 3: Import All Libraries

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from math import radians, cos, sin, asin, sqrt
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')
print(f'✅ PyTorch version: {torch.__version__}')

## STEP 4: Load Datasets

In [ ]:
# Load CSVs
print('Loading datasets...')

train_df = pd.read_csv(f'{BASE_PATH}/train_real.csv')
val_df = pd.read_csv(f'{BASE_PATH}/val_real.csv')
test_df = pd.read_csv(f'{BASE_PATH}/test_real.csv')

print(f'✅ Train: {train_df.shape}')
print(f'✅ Val: {val_df.shape}')
print(f'✅ Test: {test_df.shape}')

print(f'\n📊 Columns: {train_df.columns.tolist()}')
print(f'\n📊 Train data sample:')
print(train_df.head())

# Check ETA range
print(f'\n📊 ETA stats:')
print(f'  Min: {train_df["eta_minutes"].min():.2f} min')
print(f'  Max: {train_df["eta_minutes"].max():.2f} min')
print(f'  Mean: {train_df["eta_minutes"].mean():.2f} min')
print(f'  Std: {train_df["eta_minutes"].std():.2f} min')

## STEP 5: Load Road Graph

In [ ]:
print('Loading road graph...')

# Try to load as pickle first
try:
    graph_path = f'{BASE_PATH}/navi_mumbai_road_graph.pkl'
    if os.path.exists(graph_path):
        with open(graph_path, 'rb') as f:
            G = pickle.load(f)
        print(f'✅ Loaded from pickle')
    else:
        # Try GraphML
        graph_path = f'{BASE_PATH}/navi_mumbai_road_graph.graphml.xml'
        G = nx.read_graphml(graph_path)
        print(f'✅ Loaded from GraphML')
except Exception as e:
    print(f'❌ Could not load graph: {e}')
    print('Looking for graph files...')
    for item in os.listdir(BASE_PATH):
        if 'graph' in item.lower():
            print(f'  - {item}')

print(f'\n📊 Graph stats:')
print(f'  Nodes: {G.number_of_nodes()}')
print(f'  Edges: {G.number_of_edges()}')
print(f'  Sample node: {list(G.nodes(data=True))[0]}')
print(f'  Sample edge: {list(G.edges(data=True))[0]}')

## STEP 6: Enrich Graph Edges & Nodes

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance between two points in km using Haversine formula"""
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return 6371 * c  # Earth radius in km

PEAK_HOURS = list(range(8, 10)) + list(range(17, 19))
MONSOON_ZONES = ['Kharghar', 'Ulwe']

print('Enriching graph edges...')

for u, v, data in G.edges(data=True):
    # Add length if missing
    if 'length' not in data and 'length_km' not in data:
        u_data = G.nodes[u]
        v_data = G.nodes[v]
        if 'y' in u_data and 'x' in u_data and 'y' in v_data and 'x' in v_data:
            dist = haversine_distance(u_data['y'], u_data['x'], v_data['y'], v_data['x'])
            data['length_km'] = dist
        else:
            data['length_km'] = 0.5
    elif 'length' in data and 'length_km' not in data:
        # Convert meters to km
        data['length_km'] = data['length'] / 1000.0
    
    # Add speed profiles
    data.setdefault('speed_kmh_normal', 35.0)
    data.setdefault('speed_kmh_peak', 20.0)
    data.setdefault('speed_kmh_night', 40.0)
    data.setdefault('is_bridge', False)
    data.setdefault('road_type', data.get('highway', 'residential'))

print('✅ Graph edges enriched')

# Check edge attributes
sample_edge = list(G.edges(data=True))[0]
print(f'\n📊 Sample edge attributes:')
for k, v in sample_edge[2].items():
    print(f'  {k}: {v}')

## STEP 7: Create Location to Node Mapping

In [ ]:
# Try to load key locations file
print('Looking for key locations file...')

location_to_node = {}

try:
    loc_df = pd.read_csv(f'{BASE_PATH}/key_locations.csv')
    print(f'✅ Loaded key_locations.csv')
    print(loc_df.head())
    
    # Map location to nearest node
    # Adjust column names based on your CSV structure
    if 'location' in loc_df.columns and 'nearest_node' in loc_df.columns:
        location_to_node = dict(zip(loc_df['location'], loc_df['nearest_node']))
    else:
        print('⚠️ Unexpected columns. Available:', loc_df.columns.tolist())
        print('Manual mapping needed')
except FileNotFoundError:
    print('⚠️ key_locations.csv not found')
    print('Creating basic zone mapping...')
    
    # Create mapping from zones to random graph nodes
    zones = ['Vashi', 'Nerul', 'Kharghar', 'Belapur', 'Airoli']
    graph_nodes = list(G.nodes())
    
    for zone in zones:
        # Pick a random node for each zone
        location_to_node[zone] = np.random.choice(graph_nodes)
    
    print(f'✅ Created zone mapping')

print(f'\n📊 Location to Node mapping:')
for loc, node in list(location_to_node.items())[:5]:
    print(f'  {loc} -> {node}')

## STEP 8: Build Routes for Each Trip

In [ ]:
def find_shortest_route(G, source_node, dest_node):
    """Find shortest path in graph by length"""
    try:
        path = nx.shortest_path(G, source=source_node, target=dest_node, weight='length_km')
        return path
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None

def enrich_trip_with_route(row, G, location_to_node):
    """Add route info to a trip"""
    
    # Determine source and destination
    # Assuming dataset has 'source_zone' and 'dest_zone' or similar
    # Adjust column names based on your dataset
    
    if 'source_zone' in row.index:
        source_zone = row['source_zone']
        dest_zone = row['dest_zone']
    else:
        # Try zone columns (one-hot encoded)
        zone_cols = [c for c in row.index if c.startswith('zone_')]
        source_zone = None
        for zcol in zone_cols:
            if row[zcol] == 1:
                source_zone = zcol.replace('zone_', '')
                break
        dest_zone = 'Vashi'  # Default destination (hospital)
    
    if source_zone is None:
        return None
    
    source_node = location_to_node.get(source_zone)
    dest_node = location_to_node.get(dest_zone)
    
    if source_node is None or dest_node is None:
        return None
    
    # Find route
    route = find_shortest_route(G, source_node, dest_node)
    
    if route is None:
        return None
    
    return {
        'source_node': source_node,
        'dest_node': dest_node,
        'route': route,
        'num_segments': len(route) - 1,
    }

print('Building routes for training data...')

valid_trips = []
for idx, (trip_idx, row) in enumerate(train_df.iterrows()):
    route_info = enrich_trip_with_route(row, G, location_to_node)
    
    if route_info is not None:
        route_info['idx'] = trip_idx
        route_info['hour'] = row.get('hour', 12)
        route_info['is_monsoon'] = row.get('is_monsoon', 0)
        route_info['eta_minutes'] = row['eta_minutes']
        route_info['ambulance_type'] = row.get('ambulance_type', 0)
        valid_trips.append(route_info)
    
    if (idx + 1) % 1000 == 0:
        print(f'  Processed {idx + 1}/{len(train_df)} trips, {len(valid_trips)} valid...')

print(f'\n✅ Valid training trips: {len(valid_trips)}/{len(train_df)}')

# Sample
if valid_trips:
    print(f'\n📊 Sample trip:')
    sample = valid_trips[0]
    print(f'  Route length: {sample["num_segments"]} segments')
    print(f'  ETA: {sample["eta_minutes"]:.2f} min')
    print(f'  Hour: {sample["hour"]}')
    print(f'  Monsoon: {sample["is_monsoon"]}')

## STEP 9: Build Segment Features

In [ ]:
def build_enhanced_segment_features(G, route, hour, is_monsoon, zone_violations=0):
    """Create ENHANCED feature vector with traffic/infrastructure penalties"""
    
    segments = []
    
    # Traffic penalty mapping (violations proxy for congestion)
    traffic_penalty_map = {
        'high': 0.70,      # 30% slower
        'medium': 0.85,    # 15% slower
        'low': 1.0         # Normal speed
    }
    
    # Bridge penalty (bottleneck)
    bridge_penalty = 0.75  # 25% slower
    
    # Monsoon zone penalty
    monsoon_zones = ['Kharghar', 'Ulwe', 'Dronagiri']
    monsoon_penalty = 0.60  # 40% slower in monsoon zones
    
    for i in range(len(route) - 1):
        u, v = route[i], route[i+1]
        
        if not G.has_edge(u, v):
            continue
        
        edge = G[u][v]
        
        # Base speed
        if hour in PEAK_HOURS:
            speed_kmh = edge.get('speed_kmh_peak', 20.0)
        elif hour >= 22 or hour <= 6:
            speed_kmh = edge.get('speed_kmh_night', 40.0)
        else:
            speed_kmh = edge.get('speed_kmh_normal', 35.0)
        
        # Apply monsoon global factor
        if is_monsoon:
            speed_kmh *= 0.75
        
        # Additional penalties
        penalties = 1.0
        
        # Bridge penalty
        if edge.get('is_bridge', False):
            penalties *= bridge_penalty
        
        # Traffic penalty (zone violations as proxy)
        if zone_violations > 40000:
            traffic_level = 'high'
        elif zone_violations > 20000:
            traffic_level = 'medium'
        else:
            traffic_level = 'low'
        penalties *= traffic_penalty_map[traffic_level]
        
        # Monsoon zone specific
        zone_name = edge.get('zone', '')
        if is_monsoon and any(z in zone_name for z in monsoon_zones):
            penalties *= monsoon_penalty
        
        final_speed = speed_kmh * penalties
        
        length_km = edge.get('length_km', 0.5)
        travel_time_min = (length_km / final_speed * 60) if final_speed > 0 else 10.0
        
        segments.append({
            'length_km': float(length_km),
            'speed_kmh': float(speed_kmh),
            'final_speed_kmh': float(final_speed),
            'travel_time_min': float(travel_time_min),
            'is_bridge': float(edge.get('is_bridge', False)),
            'is_monsoon': float(is_monsoon),
            'traffic_level': float(traffic_penalty_map[traffic_level]),
            'zone_penalty': float(penalties),
        })
    
    return segments

print('✅ Enhanced segment feature function loaded')


## STEP 9B: Enhanced Segment Features (To Beat RF)

Add real-world traffic and infrastructure constraints

In [ ]:
def build_segment_features(G, route, hour, is_monsoon, zone_violations=0):
    """Create feature vector for each segment in route"""
    
    segments = []
    
    for i in range(len(route) - 1):
        u, v = route[i], route[i+1]
        
        if not G.has_edge(u, v):
            continue
        
        edge = G[u][v]
        
        # Get speed for this time
        if hour in PEAK_HOURS:
            speed_kmh = edge.get('speed_kmh_peak', 20.0)
        elif hour >= 22 or hour <= 6:
            speed_kmh = edge.get('speed_kmh_night', 40.0)
        else:
            speed_kmh = edge.get('speed_kmh_normal', 35.0)
        
        # Apply monsoon factor
        if is_monsoon:
            speed_kmh *= 0.75
        
        # NEW: Add bridge penalty
        if edge.get('is_bridge', False):
            speed_kmh *= 0.75
        
        # NEW: Add traffic penalty based on zone violations
        if zone_violations > 40000:
            speed_kmh *= 0.70  # Heavy congestion
        elif zone_violations > 20000:
            speed_kmh *= 0.85  # Moderate congestion
        
        length_km = edge.get('length_km', 0.5)
        travel_time_min = (length_km / speed_kmh * 60) if speed_kmh > 0 else 5.0
        
        segments.append({
            'length_km': float(length_km),
            'speed_kmh': float(speed_kmh),
            'travel_time_min': float(travel_time_min),
            'is_bridge': float(edge.get('is_bridge', False)),
            'is_monsoon': float(is_monsoon),
        })
    
    return segments

print('Building segment features with traffic penalties...')

for trip in valid_trips:
    segs = build_segment_features(
        G,
        trip['route'],
        int(trip['hour']),
        bool(trip['is_monsoon']),
        zone_violations=trip.get('violations_zone', 0)  # Pass zone violations if available
    )
    trip['segment_features'] = segs

# Filter out trips with no segments
valid_trips = [t for t in valid_trips if len(t['segment_features']) > 0]

print(f'✅ Trips with segment features: {len(valid_trips)}')

# Sample
if valid_trips:
    sample = valid_trips[0]
    print(f'\n📊 Sample segments (first 3):')
    for i, seg in enumerate(sample['segment_features'][:3]):
        print(f'  Segment {i+1}: length={seg["length_km"]:.3f}km, speed={seg["speed_kmh"]:.1f}km/h')


## STEP 10: Normalize ETA Target

In [ ]:
# Get ETA statistics for normalization
eta_values = np.array([t['eta_minutes'] for t in valid_trips])
eta_mean = eta_values.mean()
eta_std = eta_values.std()

print(f'📊 ETA Statistics:')
print(f'  Mean: {eta_mean:.4f} min')
print(f'  Std: {eta_std:.4f} min')
print(f'  Min: {eta_values.min():.4f} min')
print(f'  Max: {eta_values.max():.4f} min')

## STEP 11: Convert Graph to PyTorch Geometric Format

In [ ]:
print('Converting graph to PyTorch Geometric format...')

# Create node index mapping
node_id_to_idx = {node_id: idx for idx, node_id in enumerate(G.nodes())}
num_nodes = len(node_id_to_idx)

# Build node features (lat, lon, degree)
x_list = []
for node_id in G.nodes():
    node_data = G.nodes[node_id]
    lat = float(node_data.get('y', 0.0))
    lon = float(node_data.get('x', 0.0))
    degree = float(G.degree(node_id))
    x_list.append([lat, lon, degree])

x = torch.tensor(x_list, dtype=torch.float32)

# Build edge index and attributes
edge_index_list = []
edge_attr_list = []

for u, v, data in G.edges(data=True):
    u_idx = node_id_to_idx[u]
    v_idx = node_id_to_idx[v]
    
    length_km = float(data.get('length_km', 0.5))
    speed_kmh = float(data.get('speed_kmh_normal', 35.0))
    
    # Add both directions
    edge_index_list.append([u_idx, v_idx])
    edge_index_list.append([v_idx, u_idx])
    
    edge_attr_list.append([length_km, speed_kmh])
    edge_attr_list.append([length_km, speed_kmh])

edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
edge_attr = torch.tensor(edge_attr_list, dtype=torch.float32)

# Create PyG Data object
graph_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, num_nodes=num_nodes)
print(f'✅ Converted to PyG format')
print(f'  Nodes: {graph_data.num_nodes}')
print(f'  Edges: {graph_data.num_edges}')
print(f'  Node features: {graph_data.x.shape}')
print(f'  Edge features: {graph_data.edge_attr.shape}')

## STEP 12: Create Route Encoding Function

In [ ]:
def encode_trip(trip, node_id_to_idx, eta_mean, eta_std):
    """Convert trip to model input tensors"""
    
    # Route node indices
    route_node_indices = [
        node_id_to_idx[node_id] for node_id in trip['route']
    ]
    route_nodes = torch.tensor(route_node_indices, dtype=torch.long)
    
    # Segment features tensor
    seg_feat_array = np.array([
        [
            seg['length_km'],
            seg['speed_kmh'],
            seg['travel_time_min'],
            seg['is_bridge'],
        ]
        for seg in trip['segment_features']
    ])
    segment_features = torch.tensor(seg_feat_array, dtype=torch.float32)
    
    # Trip features (hour, is_monsoon)
    trip_features = torch.tensor([
        float(trip['hour']) / 24.0,
        float(trip['is_monsoon']),
    ], dtype=torch.float32)
    
    # Normalize ETA
    eta_norm = (trip['eta_minutes'] - eta_mean) / eta_std
    eta = torch.tensor([eta_norm], dtype=torch.float32)
    
    return {
        'route_nodes': route_nodes,
        'segment_features': segment_features,
        'trip_features': trip_features,
        'eta': eta,
        'eta_raw': trip['eta_minutes'],
    }

# Test on first trip
if valid_trips:
    sample_input = encode_trip(valid_trips[0], node_id_to_idx, eta_mean, eta_std)
    print(f'✅ Sample encoded trip:')
    print(f'  Route nodes: {sample_input["route_nodes"].shape}')
    print(f'  Segment features: {sample_input["segment_features"].shape}')
    print(f'  Trip features: {sample_input["trip_features"].shape}')
    print(f'  ETA (normalized): {sample_input["eta"]}')
    print(f'  ETA (raw): {sample_input["eta_raw"]:.2f} min')

## STEP 13: Build Graph-Aware GNN Model

In [ ]:
class GraphAwareGNN(nn.Module):
    """GNN that understands road network topology and trip routes"""
    
    def __init__(self, node_feat_dim=3, seg_feat_dim=4, gcn_hidden=64, output_dim=1):
        super().__init__()
        
        # Graph processing
        self.node_encoder = nn.Linear(node_feat_dim, gcn_hidden)
        self.gcn1 = GCNConv(gcn_hidden, gcn_hidden)
        self.gcn2 = GCNConv(gcn_hidden, gcn_hidden)
        
        # Segment processing
        self.segment_encoder = nn.Sequential(
            nn.Linear(seg_feat_dim, gcn_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(gcn_hidden, gcn_hidden),
        )
        
        # Attention for segment pooling
        self.attention = nn.Sequential(
            nn.Linear(gcn_hidden, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
        
        # Trip context encoder
        self.trip_encoder = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, gcn_hidden),
        )
        
        # Final predictor
        self.predictor = nn.Sequential(
            nn.Linear(gcn_hidden * 3, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim),
        )
    
    def forward(self, graph_data, route_nodes, segment_features, trip_features):
        """Forward pass
        
        Args:
            graph_data: PyG Data object with full road network
            route_nodes: Node indices in this trip's route
            segment_features: Feature matrix for route segments
            trip_features: Time and weather context
        
        Returns:
            Predicted normalized ETA
        """
        # 1. Encode nodes from graph structure
        node_embed = self.node_encoder(graph_data.x)
        
        # 2. Apply graph convolution (learn from topology)
        node_embed = torch.relu(self.gcn1(node_embed, graph_data.edge_index))
        node_embed = torch.relu(self.gcn2(node_embed, graph_data.edge_index))
        
        # 3. Extract embeddings for this route
        route_node_embeds = node_embed[route_nodes]  # [num_nodes_in_route, hidden]
        
        # 4. Encode segment features
        segment_embed = torch.relu(self.segment_encoder(segment_features))  # [num_segments, hidden]
        
        # 5. Combine route context with segments
        # Average route nodes (excluding last) as context for each segment
        route_context = (route_node_embeds[:-1] + segment_embed) / 2.0  # [num_segments, hidden]
        
        # 6. Attention pooling: weight each segment
        attention_logits = self.attention(route_context)  # [num_segments, 1]
        attention_weights = torch.softmax(attention_logits, dim=0)  # Normalize
        
        # Weighted aggregation
        aggregated_route = torch.sum(
            route_context * attention_weights,
            dim=0,
        )  # [hidden]
        
        # 7. Encode trip features
        trip_embed = torch.relu(self.trip_encoder(trip_features))  # [hidden]
        
        # 8. Combine all contexts
        final_context = torch.cat([
            aggregated_route,
            trip_embed,
            torch.mean(route_context, dim=0),  # Average over all segments
        ])  # [hidden * 3]
        
        # 9. Predict ETA
        eta_pred = self.predictor(final_context)
        
        return eta_pred

# Test model
model = GraphAwareGNN(node_feat_dim=3, seg_feat_dim=4, gcn_hidden=64)
print(f'✅ Model created')
print(f'{model}')

## STEP 14: Training Loop

In [ ]:
# Move to device
model = model.to(device)
graph_data = graph_data.to(device)

# Setup optimizer and loss
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.L1Loss()  # MAE loss

print(f'🚀 Training Graph-Aware GNN on {len(valid_trips)} trips')
print(f'  Device: {device}')
print(f'  Learning rate: 1e-3')
print(f'  Loss: L1 (MAE)')
print(f'  Epochs: 50')
print()

best_loss = float('inf')
train_losses = []

for epoch in range(50):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    
    for trip in valid_trips:
        # Encode trip
        inp = encode_trip(trip, node_id_to_idx, eta_mean, eta_std)
        
        # Move to device
        route_nodes = inp['route_nodes'].to(device)
        segment_features = inp['segment_features'].to(device)
        trip_features = inp['trip_features'].to(device)
        eta_true = inp['eta'].to(device)
        
        # Forward
        optimizer.zero_grad()
        eta_pred = model(graph_data, route_nodes, segment_features, trip_features)
        
        # Loss
        loss = criterion(eta_pred, eta_true)
        
        # Backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    train_losses.append(avg_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d}/50 | Loss: {avg_loss:.6f}')
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), '/content/gnn_model_best.pt')

print(f'\n✅ Training complete!')
print(f'Best loss: {best_loss:.6f}')
print(f'Final loss: {train_losses[-1]:.6f}')

## STEP 15: Evaluate on Validation Set

In [ ]:
# Build validation trips similarly
print('Building validation trips...')
val_valid_trips = []
for idx, (trip_idx, row) in enumerate(val_df.iterrows()):
    route_info = enrich_trip_with_route(row, G, location_to_node)
    
    if route_info is not None:
        route_info['idx'] = trip_idx
        route_info['hour'] = row.get('hour', 12)
        route_info['is_monsoon'] = row.get('is_monsoon', 0)
        route_info['eta_minutes'] = row['eta_minutes']
        route_info['ambulance_type'] = row.get('ambulance_type', 0)
        
        segs = build_segment_features(
            G,
            route_info['route'],
            int(route_info['hour']),
            bool(route_info['is_monsoon'])
        )
        route_info['segment_features'] = segs
        
        if len(segs) > 0:
            val_valid_trips.append(route_info)

print(f'✅ Valid validation trips: {len(val_valid_trips)}')

# Evaluate
print('\nEvaluating on validation set...')
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for trip in val_valid_trips:
        inp = encode_trip(trip, node_id_to_idx, eta_mean, eta_std)
        
        route_nodes = inp['route_nodes'].to(device)
        segment_features = inp['segment_features'].to(device)
        trip_features = inp['trip_features'].to(device)
        
        eta_pred_norm = model(graph_data, route_nodes, segment_features, trip_features)
        eta_pred_denorm = eta_pred_norm.item() * eta_std + eta_mean
        
        y_true.append(inp['eta_raw'])
        y_pred.append(eta_pred_denorm)

y_true = np.array(y_true)
y_pred = np.clip(np.array(y_pred), 3, 15)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print('\n' + '='*70)
print('📊 VALIDATION RESULTS')
print('='*70)
print(f'Val MAE:  {mae:.4f} min')
print(f'Val RMSE: {rmse:.4f} min')
print(f'Val R²:   {r2:.6f}')

print(f'\n🎯 Comparison with RF/LSTM baseline:')
print(f'  RF Baseline MAE: 0.0662 min')
print(f'  LSTM MAE: 0.1007 min')
print(f'  Old GNN MAE: 0.2850 min')
print(f'  New GNN MAE: {mae:.4f} min')

if mae < 0.1:
    print(f'\n✅ TARGET ACHIEVED! GNN is now competitive!')
elif mae < 0.2:
    print(f'\n🟡 Good progress, but room for improvement')
else:
    print(f'\n⚠️ Still needs tuning')

## STEP 16: Plot Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Training loss
axes[0, 0].plot(train_losses, linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Training Loss')
axes[0, 0].set_title('Training Loss Over Epochs')
axes[0, 0].grid(True, alpha=0.3)

# Actual vs Predicted
axes[0, 1].scatter(y_true, y_pred, alpha=0.5, s=20)
axes[0, 1].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2)
axes[0, 1].set_xlabel('Actual ETA (min)')
axes[0, 1].set_ylabel('Predicted ETA (min)')
axes[0, 1].set_title('Validation: Actual vs Predicted')
axes[0, 1].grid(True, alpha=0.3)

# Residuals
residuals = y_true - y_pred
axes[1, 0].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Prediction Error (min)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Prediction Errors')
axes[1, 0].grid(True, alpha=0.3)

# Model comparison
models = ['RF', 'LSTM', 'Old GNN', 'New GNN']
mae_values = [0.0662, 0.1007, 0.2850, mae]
colors = ['green', 'blue', 'red', 'orange']
axes[1, 1].bar(models, mae_values, color=colors, alpha=0.7, edgecolor='black')
axes[1, 1].set_ylabel('Test MAE (minutes)')
axes[1, 1].set_title('Model Comparison')
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Add values on bars
for i, (m, v) in enumerate(zip(models, mae_values)):
    axes[1, 1].text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('/content/gnn_results.png', dpi=100, bbox_inches='tight')
plt.show()

print('✅ Results plot saved')

## STEP 17: Evaluate on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('/content/gnn_model_best.pt'))

# Build test trips
print('Building test trips...')
test_valid_trips = []
for idx, (trip_idx, row) in enumerate(test_df.iterrows()):
    route_info = enrich_trip_with_route(row, G, location_to_node)
    
    if route_info is not None:
        route_info['idx'] = trip_idx
        route_info['hour'] = row.get('hour', 12)
        route_info['is_monsoon'] = row.get('is_monsoon', 0)
        route_info['eta_minutes'] = row['eta_minutes']
        route_info['ambulance_type'] = row.get('ambulance_type', 0)
        
        segs = build_segment_features(
            G,
            route_info['route'],
            int(route_info['hour']),
            bool(route_info['is_monsoon'])
        )
        route_info['segment_features'] = segs
        
        if len(segs) > 0:
            test_valid_trips.append(route_info)

print(f'✅ Valid test trips: {len(test_valid_trips)}')

# Evaluate
print('\nEvaluating on test set...')
model.eval()
y_test_true = []
y_test_pred = []

with torch.no_grad():
    for trip in test_valid_trips:
        inp = encode_trip(trip, node_id_to_idx, eta_mean, eta_std)
        
        route_nodes = inp['route_nodes'].to(device)
        segment_features = inp['segment_features'].to(device)
        trip_features = inp['trip_features'].to(device)
        
        eta_pred_norm = model(graph_data, route_nodes, segment_features, trip_features)
        eta_pred_denorm = eta_pred_norm.item() * eta_std + eta_mean
        
        y_test_true.append(inp['eta_raw'])
        y_test_pred.append(eta_pred_denorm)

y_test_true = np.array(y_test_true)
y_test_pred = np.clip(np.array(y_test_pred), 3, 15)

mae_test = mean_absolute_error(y_test_true, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test_true, y_test_pred))
r2_test = r2_score(y_test_true, y_test_pred)

print('\n' + '='*70)
print('📊 TEST RESULTS')
print('='*70)
print(f'Test MAE:  {mae_test:.4f} min')
print(f'Test RMSE: {rmse_test:.4f} min')
print(f'Test R²:   {r2_test:.6f}')

print(f'\n📋 Final Model Comparison:')
print(f'  RF Baseline:   MAE = 0.0662 min')
print(f'  LSTM:          MAE = 0.1007 min')
print(f'  Old Flat GNN:  MAE = 0.2850 min')
print(f'  New Graph GNN: MAE = {mae_test:.4f} min')

if mae_test < 0.15:
    print(f'\n✅ SIGNIFICANT IMPROVEMENT ACHIEVED!')
    print(f'  Reduction: {(0.2850 - mae_test)/0.2850*100:.1f}% better than old GNN')
elif mae_test < 0.2:
    print(f'\n🟡 Good improvement, still below RF/LSTM')
else:
    print(f'\n⚠️ Model needs more work')

## STEP 18: Save Model & Results

In [ ]:
# Save model
torch.save(model.state_dict(), '/content/gnn_graph_aware_final.pt')
print('✅ Model saved to /content/gnn_graph_aware_final.pt')

# Save results summary
results_summary = f"""
GNN GRAPH-AWARE MODEL - RESULTS SUMMARY
======================================

Dataset:
  Training samples: {len(valid_trips)}
  Validation samples: {len(val_valid_trips)}
  Test samples: {len(test_valid_trips)}

Graph Structure:
  Nodes: {graph_data.num_nodes}
  Edges: {graph_data.num_edges}

Model Architecture:
  GCN layers: 2
  Hidden dimension: 64
  Attention pooling: Yes
  Segment encoding: Yes

Training:
  Epochs: 50
  Learning rate: 1e-3
  Loss function: L1 (MAE)
  Best training loss: {best_loss:.6f}

Validation Results:
  MAE: {mae:.4f} min
  RMSE: {rmse:.4f} min
  R²: {r2:.6f}

Test Results:
  MAE: {mae_test:.4f} min
  RMSE: {rmse_test:.4f} min
  R²: {r2_test:.6f}

Comparison with Baselines:
  RF Baseline MAE: 0.0662 min ✓ (best)
  LSTM MAE: 0.1007 min
  Old Flat GNN MAE: 0.2850 min
  New Graph GNN MAE: {mae_test:.4f} min
  Improvement: {(0.2850 - mae_test)/0.2850*100:.1f}% better than old GNN

Key Features:
  ✓ Uses real road network topology
  ✓ Segment-level feature encoding
  ✓ Attention-based route aggregation
  ✓ Graph convolution for spatial learning
  ✓ Time/weather context integration
"""

with open('/content/gnn_results_summary.txt', 'w') as f:
    f.write(results_summary)

print(results_summary)
print('\n✅ Results saved to /content/gnn_results_summary.txt')

## STEP 19: Download Results

In [ ]:
from google.colab import files

print('📥 Download these files from Colab:')
print('  - gnn_graph_aware_final.pt (trained model)')
print('  - gnn_results.png (results visualization)')
print('  - gnn_results_summary.txt (performance summary)')

# Download model
print('\nDownloading model...')
files.download('/content/gnn_graph_aware_final.pt')
print('✅ Model downloaded')

# Download results
print('\nDownloading results...')
files.download('/content/gnn_results.png')
files.download('/content/gnn_results_summary.txt')
print('✅ Results downloaded')

## ✅ SUMMARY

You have successfully:

1. ✅ Loaded datasets from Google Drive
2. ✅ Built real road network graph (OSM)
3. ✅ Created trip routes using graph pathfinding
4. ✅ Generated segment-level features
5. ✅ Converted to PyTorch Geometric format
6. ✅ Trained graph-aware GNN model
7. ✅ Evaluated on validation & test sets
8. ✅ Compared with RF/LSTM baselines

**Next Steps:**

If MAE is still high:
- Tune hyperparameters (learning rate, hidden dims, epochs)
- Add more segment features (violation count, bridge penalties)
- Use better location mapping (avoid random nodes)
- Generate more diverse training data

If MAE is good:
- Deploy model to production
- Write paper section on GNN improvements
- Move to routing optimization phase